In [ ]:
%%bash

pip install --upgrade 'scikit-learn<1.6.0'

In [ ]:
import pathlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn import linear_model, feature_selection, metrics
from sklearn import model_selection, pipeline, preprocessing

# Multiclass Classification

## 1. Frame the Problem

In [ ]:
metrics.accuracy_score?

## 2. Load the data

In [ ]:
%%bash

ls /kaggle/input/digit-recognizer

In [ ]:
DATA_DIR = pathlib.Path("/kaggle/input/digit-recognizer")

train_df = pd.read_csv(DATA_DIR / "train.csv")
train_features_df = train_df.drop("label", axis=1)
train_labels = train_df.loc[:, "label"]

## 3. Explore the Data

In [ ]:
train_features_df.describe()

In [ ]:
fig, axes = plt.subplots(10, 10, sharex=True, sharey=True, figsize=(15, 15))
for i in range(10):
    for j in range(10):
        m, _ = train_features_df.shape
        k = np.random.randint(m)
        img = (train_features_df.loc[k, :]
                                .to_numpy()
                                .reshape((28, 28)))
        _ = axes[i, j].imshow(img)
        _ = axes[i, j].set_title(train_labels.iloc[k])

fig.suptitle("Random MNIST images", x=0.5, y=1.05, fontsize=25)
fig.tight_layout()

In [ ]:
_ = (
    train_labels.value_counts()
                .sort_index()
                .plot(kind="bar")
)
_ = plt.xticks(rotation=-45)

## 4. Prepare the Data for ML

In [ ]:
features_preparation_pipeline = pipeline.make_pipeline(
    feature_selection.VarianceThreshold(threshold=0.0),
    preprocessing.MinMaxScaler(feature_range=(-1, 1)),
)

In [ ]:
features_preparation_pipeline

## 5. Train a Few ML Models


In [ ]:
from sklearn import linear_model, ensemble

In [ ]:
linear_model_pipeline = pipeline.make_pipeline(
    features_preparation_pipeline,
    linear_model.SGDClassifier(
        loss="log_loss",
        n_jobs=-1,
    ),
)

_ = linear_model_pipeline.fit(train_features_df, train_labels)

In [ ]:
random_forest_pipeline = pipeline.make_pipeline(
    features_preparation_pipeline,
    ensemble.RandomForestClassifier(
        n_jobs=-1,
    ),
)

_ = random_forest_pipeline.fit(train_features_df, train_labels)

### Model Evaluation

In [ ]:
linear_model_cv_predictions = model_selection.cross_val_predict(
    linear_model_pipeline,
    X=train_features_df,
    y=train_labels,
    cv=3,
    n_jobs=-1,
    verbose=1
)


In [ ]:
random_forest_cv_predictions = model_selection.cross_val_predict(
    random_forest_pipeline,
    X=train_features_df,
    y=train_labels,
    cv=3,
    n_jobs=-1,
    verbose=1
)


#### Classification Report

In [ ]:
_report = metrics.classification_report(
    train_labels,
    linear_model_cv_predictions,
)
print(_report)

In [ ]:
_report = metrics.classification_report(
    train_labels,
    random_forest_cv_predictions,
)
print(_report)

#### Confusion Matrix

In [ ]:
_ = (
    metrics.ConfusionMatrixDisplay
           .from_predictions(
                train_labels,
                random_forest_cv_predictions,
            )
)

In [ ]:
# normalizing by "true" (i.e., rows) gives recalls along the diagonal!
_ = (
    metrics.ConfusionMatrixDisplay
           .from_predictions(
                train_labels,
                random_forest_cv_predictions,
                normalize="true",
                values_format=".0%"
            )
)

In [ ]:
prediction_errors = (train_labels != random_forest_cv_predictions)

_ = (
    metrics.ConfusionMatrixDisplay
           .from_predictions(
                train_labels,
                random_forest_cv_predictions,
                normalize="true",
                sample_weight=prediction_errors,
                values_format=".0%"
            )
)

In [ ]:
# normalizing by "pred" (i.e., cols) gives precision along the diagonal!
_ = (
    metrics.ConfusionMatrixDisplay
           .from_predictions(
                train_labels,
                random_forest_cv_predictions,
                normalize="pred",
                values_format=".0%"
            )
)

In [ ]:
_ = (
    metrics.ConfusionMatrixDisplay
           .from_predictions(
                train_labels,
                random_forest_cv_predictions,
                normalize="pred",
                sample_weight=prediction_errors,
                values_format=".0%"
            )
)

#### ROC AUC

In [ ]:
metrics.roc_auc_score?

In [ ]:
random_forest_cv_probas = model_selection.cross_val_predict(
    random_forest_pipeline,
    X=train_features_df,
    y=train_labels,
    cv=3,
    method="predict_proba",
    n_jobs=-1,
    verbose=1
)


In [ ]:
metrics.roc_auc_score(
    train_labels,
    random_forest_cv_probas,
    average="weighted",
    multi_class="ovr",
)

## 6. Fine-tuning an ML Model

In [ ]:
model_selection.GridSearchCV?

In [ ]:
random_forest_pipeline

In [ ]:
random_forest_pipeline.named_steps

In [ ]:
feature_selection.VarianceThreshold?

In [ ]:
ensemble.RandomForestClassifier?

In [ ]:
_param_grid = [
    {
        "pipeline__variancethreshold__threshold": [0.0, 0.1, 1.0],
        "randomforestclassifier__n_estimators": [10, 100, 1000],
        "randomforestclassifier__max_depth": [1, 10, None],
    },
]

fine_tuning_pipeline = model_selection.GridSearchCV(
    random_forest_pipeline,
    param_grid=_param_grid,
    cv=3,
    n_jobs=-1,
    refit=True,
    scoring="accuracy",
    verbose=2,
)

In [ ]:
fine_tuning_pipeline

In [ ]:
_ = fine_tuning_pipeline.fit(train_features_df, train_labels)

In [ ]:
# accessing the best hyperparameters
fine_tuning_pipeline.best_params_

In [ ]:
# accessing the score for the best hyperparameters
fine_tuning_pipeline.best_score_

In [ ]:
# accessing the best estimator (already retrained on the full training set!)
fine_tuning_pipeline.best_estimator_

## 7. Submitting to the Competition

### Generate Test Predictions

In [ ]:
test_features_df = pd.read_csv(DATA_DIR / "test.csv")
test_predictions = fine_tuning_pipeline.predict(test_features_df)


### Create Submission File

In [ ]:
_ = (
    pd.read_csv(
        "/kaggle/input/digit-recognizer/sample_submission.csv",
        index_col="ImageId"
    ).assign(
        Label=test_predictions
    ).to_csv("submission.csv", index=True)
)

In [ ]:
%%bash

cat submission.csv | head -n 5